# Scrape NvvP website voor vergoedingen

In [ ]:
import json
from pathlib import Path
from copy import deepcopy

from selenium import webdriver
from bs4 import BeautifulSoup

## Configuration

In [ ]:
SELECTED_YEAR = "2026"

YEAR_CONFIGS = {
    "2023": {
        "url": "https://www.podotherapie.nl/vergoedingen/",
        "article_class": None,
    },
    "2024": {
        "url": "https://www.podotherapie.nl/vergoedingen/",
        "article_class": None,
    },
    "2025": {
        "url": "https://www.podotherapie.nl/vergoedingen/",
        "article_class": None,
    },
    "2026": {
        "url": "https://www.podotherapie.nl/vergoedingen2026",
        "article_class": "Article--collapsible",
    },
}

config = YEAR_CONFIGS[SELECTED_YEAR]

## Scrape data

In [ ]:
options = webdriver.ChromeOptions()
options.add_argument("headless")

driver = webdriver.Chrome(options=options)
driver.get(config["url"])

soup = BeautifulSoup(driver.page_source, "html.parser")

In [ ]:
if config["article_class"]:
    articles = soup.find_all("article", class_=config["article_class"])
else:
    articles = soup.find_all("article")[1].find_all("article")

print(f"Found {len(articles)} articles on page!")

In [ ]:
def html_to_json(part):
    rows = part.find_all("tr")

    headers = {}
    thead = part.find("thead")
    if thead:
        thead = thead.find_all("th")
        for i in range(len(thead)):
            headers[i] = thead[i].text.strip().lower()
    data = []
    for row in rows:
        cells = row.find_all("td")
        if thead:
            items = {}
            if len(cells) > 0:
                for index in headers:
                    items[headers[index]] = cells[index].text
        else:
            items = []
            for index in cells:
                items.append(index.text.strip())
        if items:
            data.append(items)

    return data


providers = {}
for article in articles:
    button = article.find("button")
    if button is None:
        continue
    provider_name = button.text
    table = article.find("table")
    if table:
        products = html_to_json(table)
        providers[provider_name] = {
            "has_table": True,
            "products": products,
        }
    else:
        p_tag = article.find("p")
        if p_tag is None:
            continue
        info = p_tag.text
        providers[provider_name] = {
            "has_table": False,
            "info": info,
        }

print(f"Restructured articles to {len(providers)} insurance providers!")

## Save to data folder

In [ ]:
table_press = [
    {
        "verzekeraar": "Verzekeraar",
        "pakket": "Pakket",
        "vergoeding": "Vergoeding",
    }
]

for provider, data in providers.items():
    if provider == "a.s.r.":
        products = deepcopy(data["products"])
        info = products.pop()
        additional_info = f"\n{info[0]}: {info[1]}"
        for product in products:
            table_press.append(
                {
                    "verzekeraar": provider,
                    "pakket": product[0],
                    "vergoeding": product[1] + additional_info,
                }
            )

    elif data["has_table"]:
        for product in data["products"]:
            table_press.append(
                {
                    "verzekeraar": provider,
                    "pakket": product[0],
                    "vergoeding": product[1],
                }
            )
    else:
        table_press.append(
            {
                "verzekeraar": provider,
                "pakket": "-",
                "vergoeding": data["info"],
            }
        )

data_path = Path("../data/")
if not data_path.exists():
    data_path.mkdir()

with open(data_path / f"reimbursements_{SELECTED_YEAR}.json", "w") as f:
    json.dump(table_press, f, indent=4)

In [ ]:
print(f"Saved {len(table_press) - 1} entries for year {SELECTED_YEAR}")
table_press[:5]

In [ ]:
from collections import defaultdict

all_years_data = {}
for year in YEAR_CONFIGS.keys():
    json_file = data_path / f"reimbursements_{year}.json"
    if json_file.exists():
        with open(json_file, "r") as f:
            data = json.load(f)
            all_years_data[year] = data[1:]

provider_packages = defaultdict(dict)
for year, entries in all_years_data.items():
    provider_counts = defaultdict(int)
    for entry in entries:
        provider = entry["verzekeraar"]
        provider_counts[provider] += 1

    for provider, count in provider_counts.items():
        provider_packages[provider][year] = count

all_providers = sorted(provider_packages.keys())
years = sorted(all_years_data.keys())

print("Total packages per year:")
for year in years:
    total = len(all_years_data[year])
    print(f"  {year}: {total} packages")
print()

In [ ]:
print(f"{'Provider':<40} {' '.join(f'{y:>6}' for y in years)}")
print("=" * (40 + len(years) * 7))
for provider in all_providers:
    year_counts = [
        str(provider_packages[provider].get(y, "-")).center(6) for y in years
    ]
    print(f"{provider:<40} {' '.join(year_counts)}")